# Transfer Learning Code Illustration

This notebook gives code examples for `03_introduction_To_Transfer_learning.ipynb`.

It includes:

- Loading a pretrained model
- Freezing the backbone
- Replacing the classifier head
- Fine-tuning selected layers
- A compact U-Net style model skeleton


In [ ]:
import torch
from torch import nn

torch.manual_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device


## Load a Pretrained ResNet and Replace the Head


In [ ]:
# This cell needs torchvision. Pretrained weights may require internet access the first time.

from torchvision import models

weights = models.ResNet18_Weights.DEFAULT
model = models.resnet18(weights=weights)

for parameter in model.parameters():
    parameter.requires_grad = False

num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 3)
model = model.to(device)

print(model.fc)


## Preprocessing Expected by the Weights


In [ ]:
preprocess = weights.transforms()
preprocess


## Optimizer for Feature Extraction


In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.fc.parameters(), lr=0.001)

trainable_parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)
all_parameters = sum(p.numel() for p in model.parameters())

print('trainable parameters:', trainable_parameters)
print('all parameters:', all_parameters)


## Fine-Tuning the Last ResNet Block


In [ ]:
for parameter in model.layer4.parameters():
    parameter.requires_grad = True

optimizer = torch.optim.Adam([
    {'params': model.layer4.parameters(), 'lr': 1e-5},
    {'params': model.fc.parameters(), 'lr': 1e-3},
])

trainable_parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)
print('trainable parameters after unfreezing layer4:', trainable_parameters)


## Training Loop Skeleton


In [ ]:
# Assumes train_loader and val_loader return image tensors and class labels.

def train_one_epoch(model, train_loader, optimizer, loss_fn, device):
    model.train()
    total_loss = 0.0

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)

        logits = model(xb)
        loss = loss_fn(logits, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * xb.size(0)

    return total_loss / len(train_loader.dataset)


def evaluate(model, val_loader, loss_fn, device):
    model.eval()
    total_loss = 0.0
    correct = 0

    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = loss_fn(logits, yb)
            total_loss += loss.item() * xb.size(0)
            correct += (logits.argmax(dim=1) == yb).sum().item()

    return total_loss / len(val_loader.dataset), correct / len(val_loader.dataset)


## Compact U-Net Style Skeleton


In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class TinyUNet(nn.Module):
    def __init__(self, in_channels=3, num_classes=2):
        super().__init__()
        self.enc1 = DoubleConv(in_channels, 32)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = DoubleConv(32, 64)
        self.pool2 = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(64, 128)

        self.up2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec2 = DoubleConv(128, 64)
        self.up1 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.dec1 = DoubleConv(64, 32)

        self.out = nn.Conv2d(32, num_classes, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        b = self.bottleneck(self.pool2(e2))

        d2 = self.up2(b)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)

        return self.out(d1)


unet = TinyUNet(in_channels=3, num_classes=2)
dummy = torch.randn(2, 3, 128, 128)
mask_logits = unet(dummy)

print(mask_logits.shape)
